# DABoN: Quick Demo & Paper Figure Reproduction

This notebook demonstrates the key results of the paper using the synthetic
manifold (no GPUs required). For full benchmark results, run the experiment
scripts in `src/experiments/`.

## Contents
1. Thrust I: Negative result — geometric diversity vs BoK gain
2. Thrust II: Synthetic blind-manifold validation
3. Thrust III: DABoN selection on a toy problem
4. Paper figure reproduction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.metrics.diversity import (
    average_pairwise_cosine_distance, rnd, compute_disagreement_vectors
)
from src.algorithms.dabon import DABoNConfig, DABoNSelector
from src.experiments.thrust2_synthetic_manifold import (
    run_synthetic_experiment, plot_condition_comparison, plot_rnd_vs_geometric
)

print('Imports OK')

## 1. Thrust I Demo: Geometric Diversity is Uncorrelated with BoK Gain

We simulate a scenario where:
- A proxy reward model sees only part of the true quality
- Geometric diversity (APCD) is varied independently of reward diversity
- We show geometric diversity explains ~0% of unique variance in BoK gain

In [ ]:
from src.experiments.thrust2_synthetic_manifold import (
    sample_condition_A, sample_condition_B, sample_condition_C,
    true_quality, build_proxy_ensemble, bok_gain
)
from src.experiments.thrust1_correlation_audit import partial_correlation, unique_r2

np.random.seed(42)
K = 16
N_sets = 500
greedy_z = np.zeros(4)

rows = []
for _ in range(N_sets):
    # Mix conditions to create realistic variation
    z_set = sample_condition_A(K)
    gain  = bok_gain(z_set, greedy_z)
    scores = build_proxy_ensemble(z_set, m=3, noise_std=0.1)
    
    # Geometric diversity
    apcd_val = average_pairwise_cosine_distance(z_set)
    # RND
    rnd_val  = rnd(scores)
    # Max proxy reward
    max_r    = scores.mean(axis=1).max()
    
    rows.append({'apcd': apcd_val, 'rnd': rnd_val, 'bok_gain': gain, 'max_r': max_r})

df = pd.DataFrame(rows)

print('=== Partial Correlations (controlling for max_reward) ===')
rho_apcd = partial_correlation(df['apcd'].values, df['bok_gain'].values, df['max_r'].values)
rho_rnd  = partial_correlation(df['rnd'].values,  df['bok_gain'].values, df['max_r'].values)
ur2_apcd = unique_r2(df['apcd'].values, df['bok_gain'].values, df['max_r'].values)
ur2_rnd  = unique_r2(df['rnd'].values,  df['bok_gain'].values, df['max_r'].values)

print(f'  APCD:  partial_rho = {rho_apcd:.3f},  unique_R² = {ur2_apcd:.4f}')
print(f'  RND:   partial_rho = {rho_rnd:.3f},  unique_R² = {ur2_rnd:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, col, label, color in [
    (axes[0], 'apcd', 'Geometric Diversity (APCD)', '#ee854a'),
    (axes[1], 'rnd',  'RND (Reward-Nullspace Diversity)', '#4878d0'),
]:
    x, y = df[col].values, df['bok_gain'].values
    ax.scatter(x, y, alpha=0.25, s=10, color=color)
    m, b, r, _, _ = stats.linregress(x, y)
    xr = np.linspace(x.min(), x.max(), 100)
    ax.plot(xr, m*xr+b, 'k--', lw=1.5, label=f'r = {r:.2f}')
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel(r'BoK Gain ($\Delta_{\mathrm{BoK}}$)', fontsize=11)
    ax.legend(fontsize=10)

plt.suptitle('Figure 1: Geometric Diversity vs RND', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../figures/fig1_diversity_comparison.pdf', bbox_inches='tight')
plt.show()
print('Saved fig1')

## 2. Thrust II: Blind-Manifold Synthetic Validation

In [ ]:
df_synth = run_synthetic_experiment(n_sets=500, K=16, m=3, seed=0)

summary = df_synth.groupby('condition')['bok_gain'].agg(['mean', 'std'])
print('\nBoK Gain by Condition:')
print(summary.round(4))

from scipy.stats import f_oneway
groups = [df_synth[df_synth['condition']==c]['bok_gain'].values
          for c in ['A_full','B_observed','C_hidden']]
f, p = f_oneway(*groups)
print(f'\nOne-way ANOVA: F={f:.2f}, p={p:.2e}')

In [ ]:
import os; os.makedirs('../figures', exist_ok=True)
plot_condition_comparison(df_synth, '../figures')
plot_rnd_vs_geometric(df_synth, '../figures')
print('Figures saved.')

## 3. Thrust III: DABoN on a Toy Problem

In [ ]:
from src.algorithms.dabon import DABoNConfig, DABoNSelector, standard_bon

np.random.seed(0)
M  = 128   # pool size
m  = 3     # ensemble members
N  = 16    # selection budget

# Simulate a pool: M candidates, m reward scores each
per_model_rewards  = np.random.rand(M, m)          # (M, m)
mean_rewards       = per_model_rewards.mean(axis=1) # (M,)
disagreement       = per_model_rewards - mean_rewards[:,None]  # (M, m)

# Simulate true quality: mean reward + hidden bonus on dim 3
true_quality_vals  = mean_rewards + 0.4 * np.random.rand(M)

# Run DABoN
cfg     = DABoNConfig(budget=N, beta=1.0, adaptive_beta=False, top_k_candidates=M)
sel     = DABoNSelector(cfg)
indices = sel.select(mean_rewards, disagreement)

# Compare to standard BoN
bon_out, bon_idx = standard_bon(mean_rewards, list(range(M)), N)

dabon_best_reward = true_quality_vals[indices].max()
bon_best_reward   = true_quality_vals[bon_idx]
oracle_reward     = true_quality_vals.max()
greedy_reward     = true_quality_vals[int(np.argmax(mean_rewards))]

print(f'Greedy reward:       {greedy_reward:.4f}')
print(f'Standard BoN-{N} reward: {bon_best_reward:.4f}  (gain: {bon_best_reward-greedy_reward:.4f})')
print(f'DABoN-{N} reward:     {dabon_best_reward:.4f}  (gain: {dabon_best_reward-greedy_reward:.4f})')
print(f'Oracle reward:       {oracle_reward:.4f}')

In [ ]:
# Sample efficiency curve
budgets = [1, 2, 4, 8, 16, 32, 64]
n_trials = 200

bon_gains   = {N: [] for N in budgets}
dabon_gains = {N: [] for N in budgets}

for trial in range(n_trials):
    rng = np.random.default_rng(trial)
    per_m  = rng.random((M, m))
    mean_r = per_m.mean(axis=1)
    disag  = per_m - mean_r[:,None]
    true_q = mean_r + 0.4 * rng.random(M)
    greedy = true_q[int(np.argmax(mean_r))]
    
    for N in budgets:
        # BoN
        perm   = rng.permutation(M)[:N]
        bi     = perm[np.argmax(mean_r[perm])]
        bon_gains[N].append(true_q[bi] - greedy)
        
        # DABoN
        cfg2   = DABoNConfig(budget=N, beta=1.0, adaptive_beta=False, top_k_candidates=M)
        sel2   = DABoNSelector(cfg2)
        idxs   = sel2.select(mean_r, disag)
        best_i = idxs[int(np.argmax(mean_r[np.array(idxs)]))]
        dabon_gains[N].append(true_q[best_i] - greedy)

fig, ax = plt.subplots(figsize=(7, 4))
for label, gains_d, color, ls in [
    ('Standard BoN', bon_gains,   '#ee854a', '--'),
    ('DABoN',        dabon_gains, '#4878d0', '-'),
]:
    means = [np.mean(gains_d[N]) for N in budgets]
    sems  = [np.std(gains_d[N])/n_trials**0.5 for N in budgets]
    ax.errorbar(budgets, means, yerr=[1.96*s for s in sems],
                label=label, color=color, ls=ls, lw=2, marker='o',
                markersize=5, capsize=3)

ax.set_xlabel('Budget $N$', fontsize=12)
ax.set_ylabel(r'BoK Gain ($\Delta_{\mathrm{BoK}}$)', fontsize=12)
ax.set_xscale('log', base=2)
ax.set_xticks(budgets)
ax.set_xticklabels([str(b) for b in budgets])
ax.legend(fontsize=11)
ax.set_title('Sample Efficiency: DABoN vs Standard BoN (Toy Problem)', fontsize=11)
plt.tight_layout()
plt.savefig('../figures/fig3_efficiency_curves_toy.pdf', bbox_inches='tight')
plt.show()
print('Saved efficiency curve.')

## 4. RND Metric Validation

Show that RND is higher for diverse disagreement patterns than redundant ones.

In [ ]:
# Redundant set: all outputs have same disagreement pattern
scores_redund = np.tile([0.9, 0.5, 0.7], (8, 1))

# Diverse set: orthogonal-ish disagreement patterns
scores_diverse = np.vstack([
    [0.9, 0.5, 0.5], [0.5, 0.9, 0.5], [0.5, 0.5, 0.9],
    [0.9, 0.9, 0.5], [0.5, 0.9, 0.9], [0.9, 0.5, 0.9],
    [0.7, 0.3, 0.8], [0.3, 0.8, 0.4],
])

rnd_redund  = rnd(scores_redund)
rnd_diverse = rnd(scores_diverse)

print(f'RND (redundant patterns):  {rnd_redund:.4f}')
print(f'RND (diverse patterns):    {rnd_diverse:.4f}')
print(f'Ratio:                     {rnd_diverse / (rnd_redund + 1e-8):.2f}x')

# Visualise disagreement Gram matrices
from src.metrics.diversity import compute_disagreement_vectors
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, scores, title in [
    (axes[0], scores_redund,  f'Redundant (RND={rnd_redund:.2f})'),
    (axes[1], scores_diverse, f'Diverse (RND={rnd_diverse:.2f})'),
]:
    delta = compute_disagreement_vectors(scores)
    K_gram = delta @ delta.T
    im = ax.imshow(K_gram, cmap='RdYlGn', vmin=K_gram.min(), vmax=K_gram.max())
    plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Output index'); ax.set_ylabel('Output index')

plt.suptitle('Disagreement Gram Matrices K_Δ', fontsize=11)
plt.tight_layout()
plt.savefig('../figures/gram_matrices.pdf', bbox_inches='tight')
plt.show()